# Retinal Disease Detection Using Deep Learning on OCT Images
Classify retinal OCT scans into **CNV, DME, DRUSEN, NORMAL** with PyTorch.
Everything is in this one notebook. At the end it saves `best_model.pt`, which `streamlit_app.py` uses.

**Steps:** 1 Dataset · 2 Preprocessing · 3 Neural-network basics (NumPy) · 4 Custom CNN · 5 Transfer learning · 6 Regularization · 7 Evaluation · 8 Grad-CAM · 9 Save model + Streamlit app

> ⚠️ Educational prototype - not a clinical diagnostic tool.

**Tip for Colab:** use *Runtime → Change runtime type → GPU (T4)*. On CPU only, training is very slow; if you must use CPU, shorten `MODELS_TO_TRAIN` in Step 5.

## 0. Install and import

In [1]:
!pip install -q datasets seaborn

In [2]:
import copy, random, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             precision_recall_fscore_support)

# ---------------- SETTINGS (change these if you want) ----------------
DATASET_ID = "zacharielegault/Kermany2017-OCT"
CLASS_NAMES = ["CNV", "DME", "DRUSEN", "NORMAL"]
NUM_CLASSES = len(CLASS_NAMES)

MAX_TRAIN_PER_CLASS = 1000   # balanced subset for speed; set to None to use ALL ~108k images
VAL_FRACTION = 0.15          # share of the training subset kept for validation
IMAGE_SIZE = 224             # pretrained models expect 224 x 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

BATCH_SIZE = 32
LEARNING_RATE = 1e-3
EPOCHS = 8                   # custom CNN
WEIGHT_DECAY = 1e-4          # L2 regularization
EARLY_STOP_PATIENCE = 3
SEED = 42
NUM_WORKERS = 0              # 0 is safest on Colab / Windows

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE, "(cuda = GPU, fast | cpu = slow)")

# Make results repeatable
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

ModuleNotFoundError: No module named 'torch'

## 1. Dataset Collection and Understanding
**Kermany 2017 OCT** (Hugging Face: `zacharielegault/Kermany2017-OCT`): retinal cross-section scans with 4 classes
* **CNV** - abnormal new blood vessels under the retina
* **DME** - fluid swelling in the macula caused by diabetes
* **DRUSEN** - deposits, an early sign of age-related macular degeneration
* **NORMAL** - healthy retina

The dataset has a `train` split (108,309 images) and a `test` split (1,000 images) with columns `image` and `label`.

In [ ]:
ds = load_dataset(DATASET_ID)
print(ds)

train_data = ds["train"]
test_data = ds["test"]
print("Training samples:", len(train_data))
print("Testing samples :", len(test_data))
print(train_data.features)

label_names = train_data.features["label"].names
print("Label names:", label_names)
assert label_names == CLASS_NAMES, "Class order differs - update CLASS_NAMES above!"

In [ ]:
# Class balance. Reading only the 'label' column is fast (images are not decoded).
train_labels = np.array(train_data["label"])
test_labels = np.array(test_data["label"])

dist = pd.DataFrame({"train": np.bincount(train_labels, minlength=NUM_CLASSES),
                     "test": np.bincount(test_labels, minlength=NUM_CLASSES)}, index=CLASS_NAMES)
print(dist)
dist["train"].plot(kind="bar", color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"], rot=0)
plt.title("Training images per class (full dataset)"); plt.ylabel("Count"); plt.show()

## 2. Image Exploration and Preprocessing
Look at real examples, check image sizes, then build the preprocessing pipeline.

In [ ]:
# Show 3 examples of every class
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(10, 12))
for c in range(NUM_CLASSES):
    first_three = np.where(train_labels == c)[0][:3]
    for col, idx in enumerate(first_three):
        axes[c, col].imshow(train_data[int(idx)]["image"], cmap="gray")
        axes[c, col].set_title(CLASS_NAMES[c]); axes[c, col].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Image sizes / modes on a random sample of 200 images
rng = np.random.default_rng(SEED)
sizes, modes, broken = [], set(), 0
for i in rng.choice(len(train_data), 200, replace=False):
    try:
        img = train_data[int(i)]["image"]
        sizes.append(img.size); modes.add(img.mode)
    except Exception:
        broken += 1
print("Unreadable images in sample:", broken)
print("Image modes:", modes, "(L = grayscale; we convert to 3-channel RGB)")
print("Number of different sizes:", len(set(sizes)), "| examples:", sorted(set(sizes))[:5])
print("All images are resized to", (IMAGE_SIZE, IMAGE_SIZE))

In [ ]:
# ---- Balanced subset of the training data + validation split ----
def balanced_indices(labels, max_per_class, seed=SEED):
    """Pick up to max_per_class image positions for each class (None = take all)."""
    rng = np.random.default_rng(seed)
    chosen = []
    for c in range(NUM_CLASSES):
        idx = np.where(labels == c)[0]
        rng.shuffle(idx)
        chosen.extend(idx if max_per_class is None else idx[:max_per_class])
    chosen = np.array(chosen)
    rng.shuffle(chosen)
    return chosen

pool_idx = balanced_indices(train_labels, MAX_TRAIN_PER_CLASS)

# Validation comes from the TRAINING split only, so the test set stays completely unseen (no leakage).
# Note: patient IDs are not available in this dataset, so images of one patient may appear in both
# train and validation. The official test split (1,000 images) is the trustworthy final check.
train_idx, val_idx = train_test_split(pool_idx, test_size=VAL_FRACTION, random_state=SEED,
                                      stratify=train_labels[pool_idx])
test_idx = np.arange(len(test_data))      # use the whole official test set
print(f"Train: {len(train_idx)} | Validation: {len(val_idx)} | Test: {len(test_idx)}")

In [ ]:
# ---- Transforms: resize -> augment -> tensor -> normalize ----
normalize = transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)   # pretrained models expect this

train_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),                              # mirror
    transforms.RandomRotation(10),                                  # small tilt
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0)),    # slight zoom
    transforms.ColorJitter(brightness=0.2, contrast=0.2),           # brightness / contrast
    transforms.ToTensor(),                                          # picture -> numbers in [0,1]
    normalize,
])
eval_tf = transforms.Compose([                                      # NO augmentation for val/test
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    normalize,
])

def denormalize(tensor):
    """Undo normalization so a tensor can be displayed as a picture."""
    mean = torch.tensor(IMAGENET_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGENET_STD).view(3, 1, 1)
    return (tensor.cpu() * std + mean).clamp(0, 1)


class OCTDataset(Dataset):
    """Gives PyTorch one (image_tensor, label) pair at a time from the Hugging Face split."""

    def __init__(self, hf_split, indices, transform):
        self.hf_split, self.indices, self.transform = hf_split, indices, transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        # If an image is corrupted, try the next one instead of crashing training.
        for offset in range(len(self.indices)):
            real_idx = int(self.indices[(i + offset) % len(self.indices)])
            try:
                row = self.hf_split[real_idx]
                img = row["image"].convert("RGB")      # grayscale -> 3 channels
                return self.transform(img), int(row["label"])
            except Exception:
                continue
        raise RuntimeError("No readable images found.")


train_ds = OCTDataset(train_data, train_idx, train_tf)
val_ds = OCTDataset(train_data, val_idx, eval_tf)
test_ds = OCTDataset(test_data, test_idx, eval_tf)

# DataLoaders hand the model batches of images
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

images, labels = next(iter(train_loader))
print("One batch ->", images.shape, labels.shape)     # (batch, 3, 224, 224)

In [ ]:
# Preview: one image and 5 random augmented versions
raw = train_data[int(train_idx[0])]["image"].convert("RGB")
fig, axes = plt.subplots(1, 6, figsize=(16, 3))
axes[0].imshow(raw); axes[0].set_title("Original")
for i in range(1, 6):
    axes[i].imshow(denormalize(train_tf(raw)).permute(1, 2, 0)); axes[i].set_title(f"Augmented {i}")
for a in axes: a.axis("off")
plt.show()

## 3. Neural Network Fundamentals (NumPy)
Before PyTorch, build a tiny network by hand on the XOR puzzle to see **forward propagation, loss, backpropagation** and **gradient descent** with ReLU / Sigmoid / Tanh.

In [ ]:
def relu(x): return np.maximum(0, x)
def relu_grad(x): return (x > 0).astype(float)
def sigmoid(x): return 1 / (1 + np.exp(-x))
def sigmoid_grad(x): s = sigmoid(x); return s * (1 - s)
def tanh_grad(x): return 1 - np.tanh(x) ** 2
ACTIVATIONS = {"relu": (relu, relu_grad), "sigmoid": (sigmoid, sigmoid_grad), "tanh": (np.tanh, tanh_grad)}


class SimpleNeuralNetwork:
    """One hidden layer: input -> hidden -> output (binary)."""

    def __init__(self, n_inputs, n_hidden, activation="tanh", lr=0.5, seed=0):
        rng = np.random.default_rng(seed)
        self.act, self.act_grad = ACTIVATIONS[activation]
        self.lr = lr
        self.W1 = rng.normal(0, 0.5, (n_inputs, n_hidden)); self.b1 = np.zeros((1, n_hidden))
        self.W2 = rng.normal(0, 0.5, (n_hidden, 1));        self.b2 = np.zeros((1, 1))

    def forward(self, X):
        """Forward propagation: data flows input -> hidden -> output."""
        self.z1 = X @ self.W1 + self.b1          # weighted sum
        self.a1 = self.act(self.z1)              # activation
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = sigmoid(self.z2)               # probability 0..1
        return self.a2

    def loss(self, y, y_pred):
        """Binary cross-entropy: punishes confident wrong answers heavily."""
        eps = 1e-9
        return -np.mean(y * np.log(y_pred + eps) + (1 - y) * np.log(1 - y_pred + eps))

    def backward(self, X, y):
        """Backpropagation: chain rule gives gradients, then gradient descent updates weights."""
        m = X.shape[0]
        dz2 = self.a2 - y                                   # error at the output
        dW2 = self.a1.T @ dz2 / m; db2 = dz2.sum(0, keepdims=True) / m
        dz1 = (dz2 @ self.W2.T) * self.act_grad(self.z1)    # error sent backwards
        dW1 = X.T @ dz1 / m;       db1 = dz1.sum(0, keepdims=True) / m
        self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1

    def fit(self, X, y, epochs=2000):
        curve = []
        for _ in range(epochs):
            curve.append(self.loss(y, self.forward(X)))
            self.backward(X, y)
        return curve

    def predict(self, X):
        return (self.forward(X) > 0.5).astype(int)


X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([[0], [1], [1], [0]], dtype=float)           # XOR needs a hidden layer
for name in ACTIVATIONS:
    net = SimpleNeuralNetwork(2, 4, activation=name)
    curve = net.fit(X_xor, y_xor, epochs=3000)
    plt.plot(curve, label=f"{name} (accuracy {(net.predict(X_xor) == y_xor).mean():.2f})")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("NumPy network on XOR"); plt.legend(); plt.show()

## 4. Custom CNN (PyTorch)
Four blocks of *Conv → BatchNorm → ReLU → MaxPool*, then a small classifier with Dropout.
The same model code is used in `streamlit_app.py`.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models


class CustomCNN(nn.Module):
    """A small CNN built from scratch.
    Each block = Convolution -> BatchNorm -> ReLU -> MaxPool
      Convolution : finds patterns (edges, retinal layers, fluid pockets)
      BatchNorm   : keeps numbers stable so training is faster
      ReLU        : non-linearity, lets the network learn curved decision rules
      MaxPool     : shrinks the image, keeping the strongest signals
    """

    def __init__(self, num_classes=4, dropout=0.5):
        super().__init__()

        def block(in_ch, out_ch):
            return nn.Sequential(
                nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
                nn.BatchNorm2d(out_ch),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2),
            )

        # image size: 224 -> 112 -> 56 -> 28 -> 14
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),              # randomly switches off neurons while training
            nn.Linear(128, num_classes),      # 4 raw scores (softmax is applied later)
        )

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


def build_model(name, num_classes=4, freeze_backbone=False, pretrained=True):
    """Create any of the 5 models.
    freeze_backbone=True  -> feature extraction (only the new last layer learns)
    freeze_backbone=False -> fine-tuning (everything learns)
    """
    if name == "custom_cnn":
        return CustomCNN(num_classes)

    weights = "DEFAULT" if pretrained else None
    if name == "resnet18":
        model = models.resnet18(weights=weights)
    elif name == "vgg16":
        model = models.vgg16(weights=weights)
    elif name == "mobilenet_v2":
        model = models.mobilenet_v2(weights=weights)
    elif name == "densenet121":
        model = models.densenet121(weights=weights)
    else:
        raise ValueError("Unknown model: " + name)

    if freeze_backbone:                       # lock the pretrained knowledge
        for p in model.parameters():
            p.requires_grad = False

    # Replace the last layer (created AFTER freezing, so it is always trainable)
    if name == "resnet18":
        model.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(model.fc.in_features, num_classes))
    elif name == "vgg16":
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    else:  # mobilenet_v2, densenet121
        in_f = model.classifier[1].in_features if name == "mobilenet_v2" else model.classifier.in_features
        model.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(in_f, num_classes))
    return model

In [ ]:
def count_parameters(model):
    """(total, trainable) parameter counts."""
    return (sum(p.numel() for p in model.parameters()),
            sum(p.numel() for p in model.parameters() if p.requires_grad))


def run_one_epoch(model, loader, loss_fn, optimizer=None):
    """One pass over the data. With an optimizer = TRAIN (weights update); without = EVALUATE.
    Returns (average loss, accuracy)."""
    training = optimizer is not None
    model.train(training)
    total_loss, correct, seen = 0.0, 0, 0
    with torch.set_grad_enabled(training):
        for images, labels in loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)                 # 1. forward pass
            loss = loss_fn(outputs, labels)         # 2. how wrong are we?
            if training:
                optimizer.zero_grad()               # 3. clear old gradients
                loss.backward()                     # 4. backpropagation
                optimizer.step()                    # 5. update weights
            total_loss += loss.item() * images.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            seen += images.size(0)
    return total_loss / seen, correct / seen


def fit(model, epochs=EPOCHS, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY,
        patience=EARLY_STOP_PATIENCE, verbose=True):
    """Train with validation, L2 regularization and early stopping. Returns (model, history)."""
    model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss()
    params = [p for p in model.parameters() if p.requires_grad]       # skip frozen layers
    optimizer = torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)   # weight_decay = L2

    history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_loss, best_weights, bad_epochs = float("inf"), copy.deepcopy(model.state_dict()), 0
    start = time.time()
    for epoch in range(1, epochs + 1):
        tr_loss, tr_acc = run_one_epoch(model, train_loader, loss_fn, optimizer)
        va_loss, va_acc = run_one_epoch(model, val_loader, loss_fn)
        for k, v in zip(history, [tr_loss, va_loss, tr_acc, va_acc]):
            history[k].append(v)
        if verbose:
            print(f"Epoch {epoch:2d}/{epochs} | train loss {tr_loss:.4f} acc {tr_acc:.3f} "
                  f"| val loss {va_loss:.4f} acc {va_acc:.3f}")
        if va_loss < best_loss:                    # early stopping: remember the best epoch
            best_loss, bad_epochs = va_loss, 0
            best_weights = copy.deepcopy(model.state_dict())
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose: print(f"Early stopping at epoch {epoch}")
                break
    model.load_state_dict(best_weights)
    history["train_time_sec"] = time.time() - start
    return model, history


def plot_history(history, title):
    """A growing gap between train and validation curves = overfitting."""
    fig, ax = plt.subplots(1, 2, figsize=(10, 4))
    ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="validation")
    ax[0].set_title(f"{title} - Loss"); ax[0].set_xlabel("Epoch"); ax[0].legend()
    ax[1].plot(history["train_acc"], label="train"); ax[1].plot(history["val_acc"], label="validation")
    ax[1].set_title(f"{title} - Accuracy"); ax[1].set_xlabel("Epoch"); ax[1].legend()
    plt.tight_layout(); plt.show()

In [ ]:
custom = build_model("custom_cnn")
print("Parameters (total, trainable):", count_parameters(custom))
custom, hist_custom = fit(custom)
plot_history(hist_custom, "Custom CNN")

## 5. Transfer Learning with Pretrained Models
ResNet18, VGG16, MobileNetV2 and DenseNet121 already know edges, textures and shapes from ImageNet. We replace the last layer with a 4-class layer and train in 2 phases:
* **Phase A - Feature extraction:** freeze the backbone, train only the new layer (fast).
* **Phase B - Fine-tuning:** unfreeze everything, train with a *small* learning rate.

> VGG16 and DenseNet121 are slow on CPU. Remove them from the list below if needed.

In [ ]:
MODELS_TO_TRAIN = ["resnet18", "vgg16", "mobilenet_v2", "densenet121"]
EPOCHS_FEATURE_EXTRACTION = 3
EPOCHS_FINE_TUNING = 3

trained = {"custom_cnn": (custom, hist_custom)}
for name in MODELS_TO_TRAIN:
    print(f"\n===== {name} | Phase A: feature extraction =====")
    model = build_model(name, freeze_backbone=True)
    print("Parameters (total, trainable):", count_parameters(model))
    model, h_a = fit(model, epochs=EPOCHS_FEATURE_EXTRACTION, lr=1e-3)

    print(f"===== {name} | Phase B: fine-tuning =====")
    for p in model.parameters():
        p.requires_grad = True                       # unfreeze everything
    model, h_b = fit(model, epochs=EPOCHS_FINE_TUNING, lr=1e-4)

    hist = {k: h_a[k] + h_b[k] for k in ["train_loss", "val_loss", "train_acc", "val_acc"]}
    hist["train_time_sec"] = h_a["train_time_sec"] + h_b["train_time_sec"]
    trained[name] = (model, hist)
    plot_history(hist, name)

## 6. Optimization and Regularization
* **Overfitting:** training accuracy keeps rising while validation accuracy stalls or drops.
* **Underfitting:** both stay low.

Already used: **Dropout, L2 (weight_decay), BatchNorm, Early stopping**. Below is a small hyper-parameter experiment on the custom CNN (3 epochs each).

In [ ]:
experiments = []
for lr in [1e-3, 1e-4]:
    for wd in [0.0, 1e-4]:
        m, h = fit(build_model("custom_cnn"), epochs=3, lr=lr, weight_decay=wd, verbose=False)
        experiments.append({"lr": lr, "weight_decay": wd,
                            "best_val_acc": max(h["val_acc"]), "final_train_acc": h["train_acc"][-1]})
print(pd.DataFrame(experiments))
# Tip: also try BATCH_SIZE = 16 / 64 (re-run the DataLoader cell in Step 2 after changing it).

## 7. Model Evaluation (unseen test images)
The **test set is used only now**. Metrics: accuracy, precision, recall, F1 (macro = every class counts equally), confusion matrix, plus efficiency (parameters, training time, inference time).
The "best" model is chosen using **validation** accuracy, not test results, to avoid leaking test information.

In [ ]:
@torch.no_grad()
def predict_loader(model, loader):
    """Run the model over a DataLoader. Returns (true labels, predictions, probabilities)."""
    model.to(DEVICE).eval()
    y_true, y_pred, y_prob = [], [], []
    for images, labels in loader:
        probs = torch.softmax(model(images.to(DEVICE)), dim=1).cpu()
        y_prob.append(probs.numpy()); y_pred.extend(probs.argmax(1).tolist()); y_true.extend(labels.tolist())
    return np.array(y_true), np.array(y_pred), np.concatenate(y_prob)


def inference_ms(model, runs=20):
    """Average milliseconds to classify ONE image."""
    model.to(DEVICE).eval()
    x = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
    with torch.no_grad():
        model(x)
        start = time.time()
        for _ in range(runs): model(x)
    return (time.time() - start) / runs * 1000


results, predictions = {}, {}
for name, (model, hist) in trained.items():
    y_true, y_pred, _ = predict_loader(model, test_loader)
    predictions[name] = (y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    results[name] = {"accuracy": accuracy_score(y_true, y_pred), "precision": p, "recall": r, "f1": f1,
                     "best_val_acc": max(hist["val_acc"]),
                     "params_million": count_parameters(model)[0] / 1e6,
                     "train_time_min": hist["train_time_sec"] / 60,
                     "inference_ms": inference_ms(model)}

comparison = pd.DataFrame(results).T.sort_values("f1", ascending=False)
comparison.round(4)

In [ ]:
comparison[["accuracy", "precision", "recall", "f1"]].plot(kind="bar", figsize=(9, 4))
plt.ylim(0, 1.05); plt.title("Model comparison on the test set"); plt.ylabel("Score")
plt.xticks(rotation=20); plt.tight_layout(); plt.show()
comparison.to_csv("model_comparison.csv")

# Best model = highest VALIDATION accuracy
best_name = max(trained, key=lambda n: results[n]["best_val_acc"])
best_model = trained[best_name][0]
print("Best model (by validation accuracy):", best_name)

y_true, y_pred = predictions[best_name]
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
cm_ = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
plt.figure(figsize=(5, 4))
sns.heatmap(cm_, annot=True, fmt="d", cmap="Blues", xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion matrix - {best_name}"); plt.show()

In [ ]:
# Look at mistakes: which classes are confused and why? (blurry scan? subtle disease?)
best_model.to(DEVICE).eval()
shown, n_show = 0, 8
plt.figure(figsize=(16, 4))
with torch.no_grad():
    for images, labels in test_loader:
        preds = best_model(images.to(DEVICE)).argmax(1).cpu()
        for img, t, p in zip(images, labels, preds):
            if t != p and shown < n_show:
                plt.subplot(1, n_show, shown + 1)
                plt.imshow(denormalize(img).permute(1, 2, 0))
                plt.title(f"True: {CLASS_NAMES[t]}\nPred: {CLASS_NAMES[p]}", fontsize=8); plt.axis("off")
                shown += 1
        if shown >= n_show: break
plt.tight_layout(); plt.show()
if shown == 0: print("No misclassified images found.")

## 8. Explainability with Grad-CAM
Red areas = parts of the scan that influenced the prediction most. A good model should look at meaningful retinal structures (fluid, membranes, drusen), not at the background or image borders.

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.cm as cm


def get_target_layer(model, model_name):
    """The last convolutional stage of each architecture."""
    if model_name == "custom_cnn":
        return model.features[-1]
    if model_name == "resnet18":
        return model.layer4
    if model_name == "vgg16":
        return model.features[29]          # ReLU after the last conv layer
    if model_name == "mobilenet_v2":
        return model.features[-1]
    if model_name == "densenet121":
        return model.features
    raise ValueError("No Grad-CAM layer for " + model_name)


class GradCAM:
    """Grad-CAM: shows WHERE the model looked when it made its decision.
    1) keep the feature maps of the last conv stage  2) backpropagate from the predicted score
    3) average the gradients = importance of each map  4) weighted sum + ReLU = heat-map
    """

    def __init__(self, model, target_layer):
        self.model = model.eval()
        self.activations = None
        self.gradients = None
        self.handle = target_layer.register_forward_hook(self._save)

    def _save(self, module, inputs, output):
        out = output.clone()               # clone so in-place ReLUs cannot disturb our copy
        self.activations = out.detach().clone()
        out.register_hook(lambda grad: setattr(self, "gradients", grad.detach()))
        return out

    def remove(self):
        self.handle.remove()

    def __call__(self, input_tensor, class_idx=None):
        """input_tensor: (1, 3, H, W). Returns (heat-map HxW in [0,1], class_idx)."""
        x = input_tensor.clone().requires_grad_(True)   # needed even when the backbone is frozen
        with torch.enable_grad():
            scores = self.model(x)
            if class_idx is None:
                class_idx = int(scores.argmax(1))
            self.model.zero_grad()
            scores[0, class_idx].backward()
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=input_tensor.shape[2:], mode="bilinear", align_corners=False)[0, 0]
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy(), class_idx


def overlay_heatmap(image_rgb, heatmap, alpha=0.45):
    """Blend the heat-map on the picture. image_rgb: HxWx3 in [0,1]. Blue = low, red = high attention."""
    colored = cm.jet(heatmap)[..., :3]
    return np.clip((1 - alpha) * image_rgb + alpha * colored, 0, 1)

In [ ]:
best_cpu = copy.deepcopy(best_model).to("cpu").eval()
cam_tool = GradCAM(best_cpu, get_target_layer(best_cpu, best_name))

fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(14, 7))
for c in range(NUM_CLASSES):
    pos = int(np.where(test_labels == c)[0][0])           # first test image of class c
    tensor, _ = test_ds[pos]
    heatmap, pred_idx = cam_tool(tensor.unsqueeze(0))
    base = denormalize(tensor).permute(1, 2, 0).numpy()
    axes[0, c].imshow(base); axes[0, c].set_title(f"True: {CLASS_NAMES[c]}")
    axes[1, c].imshow(overlay_heatmap(base, heatmap)); axes[1, c].set_title(f"Pred: {CLASS_NAMES[pred_idx]}")
    axes[0, c].axis("off"); axes[1, c].axis("off")
plt.tight_layout(); plt.show()
cam_tool.remove()

## 9. Save the best model and run the Streamlit app
The checkpoint stores the weights and the model name so `streamlit_app.py` can rebuild the same model.

In [ ]:
torch.save({"model_name": best_name,
            "class_names": CLASS_NAMES,
            "state_dict": best_cpu.state_dict(),
            "test_metrics": {k: float(v) for k, v in results[best_name].items()}}, "best_model.pt")
print("Saved best_model.pt  (model:", best_name, ")")

# On Colab, download the file to your computer:
try:
    from google.colab import files
    files.download("best_model.pt")
except ImportError:
    pass

### Run the app on your computer
1. Put `best_model.pt` in the **same folder** as `streamlit_app.py`.
2. In a terminal in that folder:
```
pip install streamlit torch torchvision pillow matplotlib numpy
streamlit run streamlit_app.py
```

## Reflection (for your report)
* Which model scored best and why?  * Which two classes are confused most?  * Did Grad-CAM highlight meaningful regions?
* Limitations: subset training, one dataset, no patient IDs for patient-level splitting, not clinically validated.
* Improvements: train on all 108k images, class weights, more augmentation, ensembles, external validation.